In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [4]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'CumulativeAssetCoveredLengthKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'PeakAboveSATCount',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'Bm1Density',
        'Bm2Density',
        'B0Share',
        'B1Share',
        'Bm1Share',
        'Bm2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare',
        'POR',
        'CurrentCompletion'
    ]


In [5]:
query = f"""DROP VIEW IF EXISTS Weekly_KPI;"""
cursor.execute(query)
conn.commit()


In [6]:
query = """
CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Week'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [7]:
print(query)


CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WH

In [8]:
query = "SELECT * FROM Weekly_KPI WHERE Year = 2026 AND BoundaryRegion IS NULL;"
df = pd.read_sql_query(query, conn)
conn.close()

In [13]:
Query(f"SELECT * FROM KPI_Data WHERE BoundaryPlant IS NULL;").execute(KPIHub_Conn)

,Id,KPIId,CustomerId,BoundaryRegion,BoundaryPlant,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_Y2025_W1,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2025,Week,1.0,87.28,None,2026-09-07 09:03:58.961126
1,FOVMain_Cadent_Y2025_W2,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2025,Week,2.0,93.52,None,2026-09-07 09:03:58.961126
2,FOVMain_Cadent_Y2025_W3,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2025,Week,3.0,94.42,None,2026-09-07 09:03:58.961126
3,FOVMain_Cadent_Y2025_W4,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2025,Week,4.0,96.0,None,2026-09-07 09:03:58.961126
4,FOVMain_Cadent_Y2025_W5,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2025,Week,5.0,92.99,None,2026-09-07 09:03:58.961126
...,...,...,...,...,...,...,...,...,...,...,...
5657,DayDrivenLength_Cadent_Y2026,DayDrivenLength,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2026,Year,NaN,34.26,None,2026-09-07 09:04:13.555955
5658,NightRatio_Cadent_Y2025,NightRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2025,Year,NaN,99.97,None,2026-09-07 09:04:13.555955
5659,NightRatio_Cadent_Y2026,NightRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2026,Year,NaN,99.99,None,2026-09-07 09:04:13.555955
5660,DayRatio_Cadent_Y2025,DayRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,None,2025,Year,NaN,0.03,None,2026-09-07 09:04:13.555955
